# 🎬 குரு குல தேசம் — Upanishad 7-Scene 16:9 Video Generator (`Wan 2.1` + `LTX-Video` on A100 GPU)

This notebook generates a complete **7-Scene 16:9 Upanishad Story Video** on an **NVIDIA A100 GPU** using open-weights state-of-the-art video diffusion models (**Wan 2.1** and **LTX-Video**):

1. **Zero API Cost / Zero Flow Credits**: Runs 100% on your Colab A100 GPU in `bfloat16`.
2. **Dual Engine Support**:
   - **`LTX-Video` (`Lightricks/LTX-Video`)**: Ultra-fast (~12–18 seconds per 5-second 24fps clip on A100).
   - **`Wan 2.1` (`Wan-AI/Wan2.1-T2V-1.3B-Diffusers` or `14B`)**: Deep cinematic texture, lighting, and motion coherence (~45–75 seconds per clip on A100).
3. **Complete 7-Scene Pilot Story Embedded**:
   - Default Story: **`MUKHYA_03_KATHA_01` — கடோபநிடதம் (Katha Upanishad #003): நசிகேதனும் எமதர்மனும் — மூன்று வரங்களும் மரணமில்லாப் பெருவாழ்வும் (Nachiketa & Lord Yama)**
   - Also loads all **138 Upanishad stories (966 scenes)** automatically if your Google Drive is mounted!

👉 **How to run**:
1. Go to **Runtime → Change runtime type → Select `A100 GPU`**.
2. Click **Runtime → Run all (`Ctrl + F9`)**.

In [ ]:
# Step 1: Verify A100 GPU & Install Latest Diffusers + Video Dependencies
!pip install -q -U diffusers transformers accelerate safetensors sentencepiece protobuf imageio imageio-ffmpeg edge-tts

import torch
print("PyTorch:", torch.__version__)
assert torch.cuda.is_available(), "❌ No GPU detected! Switch Runtime -> Change runtime type -> A100 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU Active: {gpu_name} ({vram_gb:.1f} GB VRAM)")

In [ ]:
# Step 2: Mount Google Drive to save 7 scenes & assembled 1080p film directly to Drive
from pathlib import Path
import os

USE_GOOGLE_DRIVE = True
DRIVE_OUT_DIR = Path("/content/upanishad_renders")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OUT_DIR = Path("/content/drive/MyDrive/GuruKulaDesam/Videos/Upanishads_Wan21_LTX")
        DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"✅ Mounted Google Drive! Renders will be saved to: {DRIVE_OUT_DIR}")
    except Exception as e:
        print(f"⚠️ Drive mount skipped ({e}), saving locally to /content/upanishad_renders")
        DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Step 3: Select Upanishad Story & Inspect its 7 Scenes
import json

EMBEDDED_PILOT_STORIES = [{'track_id': 'MUKHYA_03_KATHA_01', 'muktika_number': 3, 'category': 'Mukhya', 'upanishad_en': 'Katha Upanishad (Part 1)', 'upanishad_ta': 'கட உபநிடதம் (பாகம் 1)', 'title_en': 'Nachiketa at the House of Death: The Boy Who Refused Kingdoms for Immortality', 'title_ta': 'எமதர்மராஜனின் வாசலில் நசிகேதன்: மரணத்தின் ரகசியத்தைக் கேட்ட சிறுவன்', 'scenes': [{'scene_num': 1, 'title': "Scene 1 (00:00 - 00:45): The Sacred Setting & The Seeker's Question (Upakrama)", 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'கட உபநிடதம்: நசிகேதனின் இதயத்தில் எழுந்த தூய சத்திய உணர்வு (ஸ்ரத்தை)!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 2, 'title': 'Scene 2 (00:45 - 01:30): The Encounter / The Trial of Qualification (Adhikara & Pariksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "'தந்தையே! என்னை யாருக்குக் கொடுப்பீர்கள்?' — 'உன்னை எமனுக்குக் கொடுக்கிறேன்!'", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 3, 'title': 'Scene 3 (01:30 - 02:15): The First Teaching / The Outer Layer of Illusion (Purvapaksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'மூன்று இரவுகள் உணவின்றி தவம் இருந்த நசிகேதனுக்கு எமதர்மன் அளித்த மூன்று வரங்கள்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 4, 'title': 'Scene 4 (02:15 - 03:00): The Deeper Inquiry & Sacred Experiment / Metaphor (Drishtanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'முதல் வரம் தந்தையின் மன அமைதி; இரண்டாம் வரம் நாசிகேத அக்னி வித்தை!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 5, 'title': 'Scene 5 (03:00 - 03:45): The Climax of Upadesha / The Mahavakya Revelation (Siddhanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'சாம்ராஜ்யங்களையும் தேவலோக போகங்களையும் தருகிறேன்; மரண ரகசியத்தை மட்டும் கேட்காதே!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 6, 'title': 'Scene 6 (03:45 - 04:30): Direct Realization & Cosmic Vision (Aparoksha Anubhuti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'நாளை அழியும் சிற்றின்பங்கள் வேண்டாம்; அழியாத ஆத்ம ஞானத்தையே அருளுங்கள்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 7, 'title': 'Scene 7 (04:30 - 05:15): The Eternal Benediction & Living Wisdom (Phala Sruti & Shanti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஸ்ரேயஸ்ச ப்ரேயஸ்ச — தீரன் நிலையற்ற இன்பத்தை விடுத்து ஆத்ம நன்மையையே தேர்ந்தெடுக்கிறான்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}]}, {'track_id': 'MUKHYA_01_ISHA_01', 'muktika_number': 1, 'category': 'Mukhya', 'upanishad_en': 'Isha Upanishad', 'upanishad_ta': 'ஈசாவாஸ்ய உபநிடதம்', 'title_en': "The Golden Disc of Pushan: The Dying Seeker's Vision of the All-Pervading Lord", 'title_ta': 'பொன்மயமான மூடியைத் திறக்கும் சத்திய தரிசனம்: ஈசாவாஸ்ய உபநிடதக் கதை', 'scenes': [{'scene_num': 1, 'title': "Scene 1 (00:00 - 00:45): The Sacred Setting & The Seeker's Question (Upakrama)", 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஈசாவாஸ்ய உபநிடதம்: இப்பிரபஞ்சத்தின் உண்மையான தலைவன் யார்?', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 2, 'title': 'Scene 2 (00:45 - 01:30): The Encounter / The Trial of Qualification (Adhikara & Pariksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஈஸா வாஸ்யமிதம் ஸர்வம் — அனைத்தும் இறைவனால் சூழப்பட்டுள்ளன.', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 3, 'title': 'Scene 3 (01:30 - 02:15): The First Teaching / The Outer Layer of Illusion (Purvapaksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'அவித்யயா ம்ருத்யும் தீர்த்வா வித்யயா அம்ருதமஸ்னுதே!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 4, 'title': 'Scene 4 (02:15 - 03:00): The Deeper Inquiry & Sacred Experiment / Metaphor (Drishtanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'யஸ்து ஸர்வாணி பூதானி ஆத்மன்யேவானுபஸ்யதி — எல்லா உயிர்களிலும் ஆத்மாவைத் தரிசிப்பவன்.', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 5, 'title': 'Scene 5 (03:00 - 03:45): The Climax of Upadesha / The Mahavakya Revelation (Siddhanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஹிரண்மயேன பாத்ரேண ஸத்யஸ்யாபிஹிதம் முகம் — பொன்மயமான மூடியைத் திறப்பாயாக!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 6, 'title': 'Scene 6 (03:45 - 04:30): Direct Realization & Cosmic Vision (Aparoksha Anubhuti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஸோஹம் அஸ்மி — அந்தப் பூரண பரம்பொருளே நான்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 7, 'title': 'Scene 7 (04:30 - 05:15): The Eternal Benediction & Living Wisdom (Phala Sruti & Shanti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஓம் பூர்ணமதஃ பூர்ணமிதம் பூர்ணாத் பூர்ணமுதச்யதே | ஓம் சாந்திஃ சாந்திஃ சாந்திஃ ||', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}]}, {'track_id': 'MUKHYA_02_KENA_01', 'muktika_number': 2, 'category': 'Mukhya', 'upanishad_en': 'Kena Upanishad', 'upanishad_ta': 'கேன உபநிடதம்', 'title_en': 'The Yaksha, the Blade of Grass & Goddess Uma Haimavati: How the Gods Lost Their Pride', 'title_ta': 'ஒற்றைப் புல்லும் உமா ஹைமவதியின் உபதேசமும்: தேவர்களின் ஆணவம் அழிந்த கதை', 'scenes': [{'scene_num': 1, 'title': "Scene 1 (00:00 - 00:45): The Sacred Setting & The Seeker's Question (Upakrama)", 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'கேன உபநிடதம்: மனதையும் பிராணனையும் உள்ளிருந்து இயக்குபவர் யார்?', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 2, 'title': 'Scene 2 (00:45 - 01:30): The Encounter / The Trial of Qualification (Adhikara & Pariksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'தேவர்களின் ஆணவத்தை அழிக்க ஒளிமயமான யக்ஷ வடிவில் தோன்றிய பிரம்மம்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 3, 'title': 'Scene 3 (01:30 - 02:15): The First Teaching / The Outer Layer of Illusion (Purvapaksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'உலகையே எரிக்கும் அக்னியால் அந்த ஒற்றைப் புல்லைக் கூட எரிக்க முடியவில்லை!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 4, 'title': 'Scene 4 (02:15 - 03:00): The Deeper Inquiry & Sacred Experiment / Metaphor (Drishtanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'மலைகளை உலுக்கும் வாயுவாலும் அந்தச் சிறு புல்லை அசைக்க முடியவில்லை!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 5, 'title': 'Scene 5 (03:00 - 03:45): The Climax of Upadesha / The Mahavakya Revelation (Siddhanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'பொன்மயமான உமா ஹைமவதி தோன்றிப் பிரம்ம தத்துவத்தை உபதேசித்தாள்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 6, 'title': 'Scene 6 (03:45 - 04:30): Direct Realization & Cosmic Vision (Aparoksha Anubhuti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'பிரம்மத்தின் வெற்றியினாலேயே நீங்கள் பெருமை பெற்றீர்கள் — உமா தேவியின் உபதேசம்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 7, 'title': 'Scene 7 (04:30 - 05:15): The Eternal Benediction & Living Wisdom (Phala Sruti & Shanti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'தவமும், புலனடக்கமும், நற்செயலுமே பிரம்ம வித்தையின் அடித்தளம்; சத்தியமே அதன் உறைவிடம்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}]}, {'track_id': 'MUKHYA_11_CHANDOGYA_01', 'muktika_number': 9, 'category': 'Mukhya', 'upanishad_en': 'Chandogya Upanishad (Story 1: Satyakama Jabala)', 'upanishad_ta': 'சாந்தோக்ய உபநிடதம் (சத்யகாம ஜாபாலன் கதை)', 'title_en': 'Satyakama Jabala & The Four-Quartered Brahman: How Fearless Truthfulness Made a True Brahmana', 'title_ta': 'சத்யகாம ஜாபாலனின் உண்மை விரதம்: காளையரும் நெருப்பும் அன்னமும் உபதேசித்த பிரம்ம வித்தை!', 'scenes': [{'scene_num': 1, 'title': "Scene 1 (00:00 - 00:45): The Sacred Setting & The Seeker's Question (Upakrama)", 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "சாந்தோக்ய உபநிடதம்: 'என் பெயர் ஜாபாலா, உன் பெயர் சத்யகாமன்; சத்யகாம ஜாபாலன் என்றே கூறு!'", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 2, 'title': 'Scene 2 (00:45 - 01:30): The Encounter / The Trial of Qualification (Adhikara & Pariksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ந ஸத்யாதகா இதி — சத்தியத்திலிருந்து விலகாதவனே உண்மையான பிராமணன்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 3, 'title': 'Scene 3 (01:30 - 02:15): The First Teaching / The Outer Layer of Illusion (Purvapaksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'நானூறு பசுக்கள் ஆயிரமாகப் பெருகும் வரை காட்டிலேயே தங்கித் தவம் இயற்றிய சத்யகாமன்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 4, 'title': 'Scene 4 (02:15 - 03:00): The Deeper Inquiry & Sacred Experiment / Metaphor (Drishtanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "காளை 'ப்ரகாசவான்' பாதத்தையும், அக்னி 'அனந்தவான்' பாதத்தையும் சத்யகாமனுக்கு உபதேசித்தன!", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 5, 'title': 'Scene 5 (03:00 - 03:45): The Climax of Upadesha / The Mahavakya Revelation (Siddhanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "அன்னப் பறவை 'ஜ்யோதிஷ்மான்' பாதத்தையும், நீர்க்கோழி 'ஆயதனவான்' பாதத்தையும் உபதேசித்தன!", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 6, 'title': 'Scene 6 (03:45 - 04:30): Direct Realization & Cosmic Vision (Aparoksha Anubhuti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "'பிரம்மத்தை அறிந்தவன் போல் உன் முகம் ஒளிவீசுகிறதே!' — குருவின் திருவாக்கால் முழுமை பெற்ற ஞானம்!", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 7, 'title': 'Scene 7 (04:30 - 05:15): The Eternal Benediction & Living Wisdom (Phala Sruti & Shanti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'சத்திய விரதமும் குரு பக்தியுமே பிரம்ம ஞானத்தின் உண்மையான தகுதி! ஓம் சாந்திஃ சாந்திஃ சாந்திஃ!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}]}, {'track_id': 'MUKHYA_12_CHANDOGYA_02', 'muktika_number': 9, 'category': 'Mukhya', 'upanishad_en': 'Chandogya Upanishad (Story 2: Svetaketu & Uddalaka Aruni)', 'upanishad_ta': 'சாந்தோக்ய உபநிடதம் (உத்தாலகர் - சுவேதகேது: தத் த்வம் அஸி)', 'title_en': "Svetaketu, the Banyan Seed & the Salt in Water: The Ninefold Revelation of 'Tat Tvam Asi'", 'title_ta': "ஆலமர விதையும் நீரில் கரைந்த உப்பும்: சுவேதகேதுவுக்கு அருளப்பட்ட 'தத் த்வம் அஸி' மகா வாக்கியம்!", 'scenes': [{'scene_num': 1, 'title': "Scene 1 (00:00 - 00:45): The Sacred Setting & The Seeker's Question (Upakrama)", 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "சாந்தோக்ய உபநிடதம்: 'எதை அறிந்தால் அறியாத அனைத்தும் அறிந்ததாகுமோ, அதைக் கேட்டாயா?'", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 2, 'title': 'Scene 2 (00:45 - 01:30): The Encounter / The Trial of Qualification (Adhikara & Pariksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'வாசாரம்பணம் விகாரோ நாமதேயம் — வடிவங்கள் வெறும் பெயர்களே; மூலப் பொருளான சத்தே உண்மை!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 3, 'title': 'Scene 3 (01:30 - 02:15): The First Teaching / The Outer Layer of Illusion (Purvapaksha)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': "'ஆலமரத்தின் நுண்ணிய விதையைப் பிளந்து பார்; உள்ளே என்ன தெரிகிறது?' — 'ஒன்றுமே தெரியவில்லை தந்தையே!'", 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 4, 'title': 'Scene 4 (02:15 - 03:00): The Deeper Inquiry & Sacred Experiment / Metaphor (Drishtanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'கண்ணுக்குத் தெரியாத அந்த நுண்ணிய சாரமே இப்பெரிய ஆலமரம் — தத் த்வம் அஸி (அதுவே நீ)!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 5, 'title': 'Scene 5 (03:00 - 03:45): The Climax of Upadesha / The Mahavakya Revelation (Siddhanta)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'நீரில் கரைந்த உப்பு கண்ணுக்குத் தெரியாவிட்டாலும் எங்கும் நிறைந்திருப்பது போல் உள்ள ஆத்மா — தத் த்வம் அஸி!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 6, 'title': 'Scene 6 (03:45 - 04:30): Direct Realization & Cosmic Vision (Aparoksha Anubhuti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'ஆசார்யவான் புருஷோ வேத — சற்குருவை அடைந்தவனே அறியாமைக் கட்டு நீங்கி மெய்ப்பொருளை அறிகிறான்!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}, {'scene_num': 7, 'title': 'Scene 7 (04:30 - 05:15): The Eternal Benediction & Living Wisdom (Phala Sruti & Shanti)', 'prompt': '', 'vo_en': '', 'vo_ta': '', 'subtitle_cue': 'தத் த்வம் அஸி ஸ்வேதகேதோ — சுவேதகேதுவே, அந்தப் பரம்பொருளே நீ! ஓம் சாந்திஃ சாந்திஃ சாந்திஃ!', 'negative_prompt': 'No text, no subtitles, no watermarks, no modern clothing, no western clothes, no electric wires, no plastic, no anime, no morphing limbs'}]}]

# Choose which story to generate (0 = Katha Upanishad: Nachiketa & Yama, 1 = Isha, 2 = Kena, 3 = Chandogya Satyakama, 4 = Chandogya Tat Tvam Asi)
STORY_INDEX = 0

# Or if you uploaded google_flow_7scene_master_manifest.json to Drive, load all 138 stories automatically:
manifest_drive_path = Path("/content/drive/MyDrive/GuruKulaDesam/upanishad_stories/google_flow_7scene_master_manifest.json")
if manifest_drive_path.exists():
    all_tracks = json.load(open(manifest_drive_path, encoding="utf-8"))["tracks"]
    print(f"📚 Loaded full 138-Story Master Manifest from Drive ({len(all_tracks)} tracks)!")
    selected_story = all_tracks[STORY_INDEX]
else:
    selected_story = EMBEDDED_PILOT_STORIES[STORY_INDEX]

print("=================================================================")
print(f"🎬 SELECTED STORY: {selected_story['track_id']} (Muktika #{selected_story['muktika_number']:03d})")
print(f"📖 Upanishad     : {selected_story['upanishad_en']} | {selected_story['upanishad_ta']}")
print(f"🌟 Title (EN)    : {selected_story['title_en']}")
print(f"🌟 Title (TA)    : {selected_story['title_ta']}")
print("=================================================================")
for sc in selected_story["scenes"]:
    print(f"  • Scene {sc['scene_num']}: {sc['title']}")

In [ ]:
# Step 4: Load Video Generation Engine on A100 (`wan2.1` or `ltx` or `both_scene1_compare`)
# Options:
#   "wan2.1_1.3b" -> Wan-AI/Wan2.1-T2V-1.3B-Diffusers (Fast + High Cinematic Quality, ~14GB VRAM)
#   "wan2.1_14b"  -> Wan-AI/Wan2.1-T2V-14B-Diffusers  (Maximum 14B Quality on A100 40GB/80GB with CPU offload)
#   "ltx"         -> Lightricks/LTX-Video             (Blazing Fast ~15s per clip on A100)
#   "compare_then_wan" -> Generates Scene 1 in BOTH LTX-Video and Wan 2.1 first so you can compare, then generates Scenes 1..7!

ENGINE_MODE = "compare_then_wan"  # Change to "ltx", "wan2.1_1.3b", or "wan2.1_14b" if desired

import gc
import torch
from diffusers import DiffusionPipeline, LTXPipeline, AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

def load_ltx_pipeline():
    print("⏳ Loading Lightricks/LTX-Video in bfloat16 on A100...")
    pipe = LTXPipeline.from_pretrained(
        "Lightricks/LTX-Video",
        torch_dtype=torch.bfloat16
    ).to("cuda")
    print("✅ LTX-Video loaded on A100!")
    return pipe

def load_wan_pipeline(model_id="Wan-AI/Wan2.1-T2V-1.3B-Diffusers"):
    print(f"⏳ Loading {model_id} in bfloat16 on A100...")
    vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(
        model_id,
        vae=vae,
        torch_dtype=torch.bfloat16
    )
    if "14B" in model_id:
        pipe.enable_model_cpu_offload()
    else:
        pipe.to("cuda")
    print(f"✅ {model_id} loaded on A100!")
    return pipe

def generate_clip_ltx(pipe, prompt, negative_prompt, out_path, seed=42):
    generator = torch.Generator("cuda").manual_seed(seed)
    video = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=768,
        height=512,
        num_frames=121,          # 5.0 seconds @ 24fps
        num_inference_steps=40,
        guidance_scale=3.5,
        generator=generator,
    ).frames[0]
    export_to_video(video, str(out_path), fps=24)

def generate_clip_wan(pipe, prompt, negative_prompt, out_path, seed=42):
    generator = torch.Generator("cuda").manual_seed(seed)
    video = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=832,
        height=480,              # Native 16:9 widescreen Wan 2.1 resolution
        num_frames=81,           # 5.0 seconds @ 16fps (upsampled to 24fps 1080p in post)
        num_inference_steps=35,
        guidance_scale=5.0,
        generator=generator,
    ).frames[0]
    export_to_video(video, str(out_path), fps=16)

In [ ]:
# Step 5: Generate the 7 Scenes for the Selected Upanishad Story
import gc, time, json
from pathlib import Path
from IPython.display import HTML, display
from base64 import b64encode

# Self-healing fallback if Step 2 or Step 3 was skipped:
if "DRIVE_OUT_DIR" not in globals():
    DRIVE_OUT_DIR = Path("/content/drive/MyDrive/GuruKulaDesam/Videos/Upanishads_Wan21_LTX") if Path("/content/drive/MyDrive").exists() else Path("/content/upanishad_renders")
    DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

if "selected_story" not in globals():
    if "EMBEDDED_PILOT_STORIES" in globals():
        selected_story = EMBEDDED_PILOT_STORIES[0]
    else:
        raise RuntimeError("Please run Step 3 cell above first (or click Runtime -> Run all) so EMBEDDED_PILOT_STORIES is loaded!")

story_dir = DRIVE_OUT_DIR / selected_story["track_id"]
story_dir.mkdir(parents=True, exist_ok=True)

def show_video_inline(mp4_path, label):
    mp4 = open(mp4_path, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"<h4>{label} ({mp4_path.name})</h4><video width=640 controls autoplay loop muted><source src='{data_url}' type='video/mp4'></video>"))

# 1. Optional Side-by-Side Scene 1 Benchmark (LTX-Video vs Wan 2.1)
if ENGINE_MODE == "compare_then_wan":
    sc1 = selected_story["scenes"][0]
    ltx_test_path = story_dir / "scene_01_LTX_benchmark.mp4"
    if not ltx_test_path.exists():
        ltx_pipe = load_ltx_pipeline()
        t0 = time.time()
        generate_clip_ltx(ltx_pipe, sc1["prompt"], sc1["negative_prompt"], ltx_test_path, seed=101)
        print(f"⚡ LTX-Video generated Scene 1 in {time.time() - t0:.1f}s -> {ltx_test_path}")
        del ltx_pipe
        gc.collect()
        torch.cuda.empty_cache()
    show_video_inline(ltx_test_path, "⚡ LTX-Video Benchmark — Scene 1")

# 2. Generate all 7 Scenes (scene_01.mp4 .. scene_07.mp4)
if ENGINE_MODE == "ltx":
    active_pipe = load_ltx_pipeline()
    gen_fn = generate_clip_ltx
elif ENGINE_MODE == "wan2.1_14b":
    active_pipe = load_wan_pipeline("Wan-AI/Wan2.1-T2V-14B-Diffusers")
    gen_fn = generate_clip_wan
else:
    active_pipe = load_wan_pipeline("Wan-AI/Wan2.1-T2V-1.3B-Diffusers")
    gen_fn = generate_clip_wan

for sc in selected_story["scenes"]:
    snum = sc["scene_num"]
    out_mp4 = story_dir / f"scene_{snum:02d}.mp4"
    if out_mp4.exists():
        print(f"✅ Scene {snum:02d} already exists at {out_mp4}, skipping.")
        continue
    print(f"🎬 Generating Scene {snum:02d}/07: {sc['title']}...")
    t0 = time.time()
    gen_fn(active_pipe, sc["prompt"], sc["negative_prompt"], out_mp4, seed=100 + snum)
    print(f"   ✅ Saved {out_mp4.name} in {time.time() - t0:.1f}s")
    if snum == 1:
        show_video_inline(out_mp4, f"🌟 Scene 01 ({ENGINE_MODE})")

del active_pipe
gc.collect()
torch.cuda.empty_cache()
print(f"🎉 All 7 scenes generated in: {story_dir}")

In [ ]:
# Step 6: Generate Neural Voiceover (Tamil or English), Extend Scenes & Assemble 1080p Master Film
import asyncio
import subprocess
import edge_tts

LANGUAGE = "ta"  # Set "ta" for Senthamizh voiceover (Valluvar) or "en" for English narration
VOICE_MAP = {
    "ta": "ta-IN-ValluvarNeural",
    "en": "en-IN-PrabhatNeural"
}

async def synthesize_scene_audio():
    voice = VOICE_MAP[LANGUAGE]
    for sc in selected_story["scenes"]:
        snum = sc["scene_num"]
        text = sc["vo_ta"] if LANGUAGE == "ta" else sc["vo_en"]
        audio_path = story_dir / f"vo_{snum:02d}_{LANGUAGE}.mp3"
        if not audio_path.exists():
            communicate = edge_tts.Communicate(text, voice, rate="-8%")
            await communicate.save(str(audio_path))
            print(f"🎙️ Synthesized Scene {snum:02d} voiceover ({LANGUAGE}) -> {audio_path.name}")

await synthesize_scene_audio()

# Build 2-Step Ping-Pong 1080p Loop + Multiplex Voiceover for all 7 Scenes
segment_files = []
for sc in selected_story["scenes"]:
    snum = sc["scene_num"]
    raw_mp4 = story_dir / f"scene_{snum:02d}.mp4"
    vo_mp3 = story_dir / f"vo_{snum:02d}_{LANGUAGE}.mp3"
    pingpong_mp4 = story_dir / f"pingpong_{snum:02d}.mp4"
    seg_mp4 = story_dir / f"segment_{snum:02d}_1080p.mp4"

    # Step 6A: Create a single 12-second seamless ping-pong unit (forward + reverse) on the short 5s clip
    fc = (
        "[0:v]scale=1920:1080:flags=lanczos,fps=24,format=yuv420p,split=2[v1][v2];"
        "[v1]setpts=1.25*PTS[va];"
        "[v2]reverse,setpts=1.25*PTS[vb];"
        "[va][vb]concat=n=2:v=1:a=0[vloop]"
    )
    subprocess.run([
        "ffmpeg", "-y", "-i", str(raw_mp4),
        "-filter_complex", fc,
        "-map", "[vloop]",
        "-c:v", "libx264", "-preset", "fast", "-crf", "18", "-an",
        str(pingpong_mp4)
    ], check=True)

    # Step 6B: Infinitely loop the ping-pong unit (-stream_loop -1) to match the exact voiceover audio length
    subprocess.run([
        "ffmpeg", "-y",
        "-stream_loop", "-1", "-i", str(pingpong_mp4),
        "-i", str(vo_mp3),
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "copy",
        "-c:a", "aac", "-b:a", "192k",
        "-shortest",
        str(seg_mp4)
    ], check=True)

    segment_files.append(seg_mp4)
    print(f"🎞️ Built 1080p synchronized segment {snum:02d} -> {seg_mp4.name}")

# Concatenate all 7 segments into the final 1080p Master Film
concat_txt = story_dir / "concat_list.txt"
concat_txt.write_text("\n".join([f"file '{p.name}'" for p in segment_files]), encoding="utf-8")

final_film = story_dir / f"{selected_story['track_id']}_{LANGUAGE}_Master_1080p.mp4"
subprocess.run([
    "ffmpeg", "-y", "-f", "concat", "-safe", "0",
    "-i", str(concat_txt),
    "-c", "copy",
    str(final_film)
], check=True)

print("=================================================================")
print(f"🏆 MASTER 1080p 7-SCENE FILM READY: {final_film}")
print("=================================================================")
show_video_inline(final_film, f"🏆 Full 7-Scene Upanishad Film — {selected_story['title_en']}")